### <b>Phase 1: Data Collection</b>

<b>Task 1.1 - Books to Scrape (HTML, Requests, BeautifulSoup)</b>

- Retrieve the first three catalogue pages (approximately 60 books total when all pages are accessible).
- Use Requests with a timeout and status checks; parse the HTML using BeautifulSoup.
- Extract title, displayed GBP price, star rating, availability status, absolute product URL, and page number.
- Demonstrate pagination using a loop, and handle missing tags/attributes without crashing.
- Export the raw results to data/raw/books_raw.csv and preserve the extraction timestamp

In [112]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
from datetime import datetime, timezone


rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
books_clean = []
books_raw = []


for page in range(1,4):

    try:
        url = f"https://books.toscrape.com/catalogue/page-{page}.html" 
        
        r = requests.get(
            url,
            timeout=15
        )
        r.raise_for_status()
        r.encoding = r.apparent_encoding
        
        
        soup = BeautifulSoup(r.text, "html.parser")
        books = soup.select(".product_pod")
        
        
        for book in books:
           
           
           title = book.select_one("h3 a")["title"] if title else None
           
         
           
           price = book.select_one("p.price_color").get_text(strip=True) if price else None
           price_clean = float(price.replace("Â", "").replace("£", "")) if price_clean else None
           
           rating = book.select_one("p.star-rating")["class"][1] if rating else None
           
           rating_num = rating_map[rating]

           availability = book.select_one("p.instock.availability").get_text(strip=True) if availability else None
           
           relative_url = book.select_one("h3 a")["href"] if relative_url else None
           
           abs_url = urljoin(url, relative_url)
           
           
               
           books_clean.append({
                     "Title": title,
                     "Price": price_clean,
                     "Rating": rating_num,
                     "availability": availability,
                     "AbsoluteUrl": abs_url,
                     "Page": page 
                 })
           
           
           books_raw.append({
                                "Title": title,
                                "Price": price,
                                "Rating": rating,
                                "availability": availability,
                                "AbsoluteUrl": abs_url,
                                "Page": page 
                            })
                      
           
                     
           
    except requests.RequestException as e:
        print(f"Error: {e} in page {page}")
        


extraction_time = datetime.now(timezone.utc).isoformat()
print("Data extracted at: ", extraction_time)

df_raw = pd.DataFrame(books_raw)
df_raw.to_csv("../data/raw/books_raw.csv", index=False)

df = pd.DataFrame(books_clean)
df.to_csv("../data/clean/books_clean.csv", index=False)



Data extracted at:  2026-11-04T05:00:03.519017+00:00


<b>Task 1.2 - Fake Store API (REST, JSON)</b>

- Send an HTTP GET request to https://fakestoreapi.com/products.
- Check HTTP status and parse JSON; verify the top-level structure and number of records.
- Select ID, title, price, category, and rating fields (including nested rating information).
- Export a raw JSON copy to data/raw/products_raw.json and a CSV snapshot if desired

In [113]:
import json
import requests
import pandas as pd

# ---------the site didin't work for me becaue it requires VPN so I've downloaded the file--------------
# url = "https://fakestoreapi.com/products"

# r = requests.get(
#     url,
#     timeout=15
# )

# r.raise_for_status()
# r.json()

# with open("../data/raw/products_raw.json", "w", encoding="utf-8") as file:
#     json.dump(r.text, file, indent=4) 

with open("../data/raw/products_raw.json", "r", encoding="utf-8-sig") as f:
    products = json.load(f)
    
extraction_time = datetime.now(timezone.utc).isoformat()
print("Data extracted at: ", extraction_time)


print(len(products))  # No. of records

# products_list = []

# for product in products:    

#     new_product = {
#         "id": product["id"],
#         "title": product["title"],
#         "price": product["price"],
#         "category": product["category"],
#         "rating_rate": product["rating"]["rate"],
#         "rating_count": product["rating"]["count"],
#         }
    
#     products_list.append(new_product)
    

df = pd.json_normalize(products)

df.to_csv("../data/raw/products_raw.csv", index=False)


df = df[['id', 'title', 'price', 'category', 'rating.rate', 'rating.count']]

df = df.rename(columns=
    {
        "rating.rate": "rating_rate",
        "rating.count": "rating_count"
    }
)


product_dict = df.to_dict(orient="records")

    
with open("../data/clean/products_clean.json", "w", encoding="utf-8-sig") as file:
    json.dump(product_dict, file, indent=4)
    


df.to_csv("../data/clean/products_clean.csv", index=False)


Data extracted at:  2026-11-04T05:00:05.048588+00:00
20


<b>Task 1.3 - Amazon Stock History (yfinance)</b>

- Create a yfinance Ticker object using AMZN.
- Request the latest available 1-year historical series at daily intervals.
- Reset the date index safely, inspect column names, and preserve OHLCV fields.
- Save the historical extract as data/raw/amzn_raw.csv.
- Document whether automatic price adjustment was enabled and how this affects Open, High, Low, and Close.


In [114]:
import yfinance as yf


amzn = yf.Ticker("AMZN")

df = amzn.history(period="1y", interval="1d", auto_adjust=False)

extraction_time = datetime.now(timezone.utc).isoformat()
print("Data extracted at: ", extraction_time)

"""
I wrote (auto_adjust=False), which means that yfinance doesn't apply
its additional auto adj to the OLHC columns, Adj Close is povided 
as a seperate column

"""

if df.empty:
    print("No Data returned")
    
else:
    df.reset_index(inplace=True)
    
    required_cols = ['Date','Open', 'High', 'Low', 'Close', 'Volume']
    
    if all(col in df.columns for col in required_cols):
        print("All required Columns exists!")

        df.to_csv("../data/raw/amzn_raw.csv", index=False)

        df = df[required_cols]

        df.to_csv("../data/clean/amzn_clean.csv", index=False)
        
    else:
        print("Some required columns are missing!")
        










Data extracted at:  2026-11-04T05:00:23.424422+00:00
All required Columns exists!


d:\Projects\retail_intelligence_data_analysis\.venv\Lib\site-packages\yfinance\utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
C:\Users\Resala\AppData\Local\Temp\ipykernel_5164\211038158.py:22: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.reset_index(inplace=True)


### <b>Phase 2 - Data Validation and Cleaning</b>


In [ ]:
amzn_clean = pd.read_csv("../data/clean/amzn_clean.csv")
products_clean = pd.read_csv("../data/clean/products_clean.csv")
book_clean = pd.read_csv("../data/clean/books_clean.csv")



